# 03 · Financial Viability

Assesses whether the 20 recommended 0.5 GW wind farms are economically viable, computing **NPV, IRR and LCOE** per site and a price × CapEx **sensitivity analysis**.

**Key assumptions:** 500 MW/farm · 35 yr · capacity factor 33% (England) / 38.1% (Scotland, Wales, NI) · discount rate 5.8% · electricity price £73.8/MWh (CfD AR6) · CapEx £1.588 M/MW · OpEx £40.1k/MW/yr · grid £2.03 M/MW·km.

## NPV, IRR & LCOE per site
Full discounted cash-flow model for each of the 20 sites (35-year lifetime, 5.8% discount rate), including site-specific grid-connection cost as a function of distance to the high-voltage network.

In [9]:
import numpy as np
import pandas as pd
import numpy_financial as npf

# =============================================================================
# 1. PARAMETER AUS TABELLE 6 & STANDORT-DATEN
# =============================================================================

distances_km = [
    0.466, 3.116, 2.799, 0.875, 3.586, 8.089, 1.875, 4.581, 4.736, 5.990,
    7.085, 24.955, 14.071, 6.363, 26.411, 19.926, 9.122, 7.773, 19.966, 17.263
]

capacity_mw = 500
lifetime = 35
cap_factor = 0.381
discount_rate = 0.058
elec_price = 73.8

capex_mw = 1588000
opex_mw_a = 40100
grid_cost_km = 2030000

# =============================================================================
# 2. BERECHNUNG (VOLLSTÄNDIGE DISKONTIERUNG)
# =============================================================================

results = []

for i, dist in enumerate(distances_km):
    # 1. Investitionskosten (t=0)
    total_capex = (capacity_mw * capex_mw) + (dist * grid_cost_km)
    
    # 2. Jährliche Basiswerte
    annual_gen = capacity_mw * 8760 * cap_factor
    annual_revenue = annual_gen * elec_price
    annual_opex = opex_mw_a * capacity_mw
    annual_net_cashflow = annual_revenue - annual_opex
    
    # 3. Diskontierung für LCOE
    # Wir diskontieren die Energieproduktion (E) und die Betriebskosten (OpEx)
    # Formel: Summe [ Wert / (1 + r)^t ]
    years = np.arange(1, lifetime + 1)
    discount_factors = (1 + discount_rate) ** years
    
    discounted_opex_sum = np.sum(annual_opex / discount_factors)
    discounted_gen_sum = np.sum(annual_gen / discount_factors)
    
    # 4. LCOE Berechnung
    # LCOE = (CapEx + Summe diskontierter OpEx) / Summe diskontierter Energie
    lcoe = (total_capex + discounted_opex_sum) / discounted_gen_sum
    
    # 5. NPV und IRR (Netto-Cashflow-Betrachtung)
    # Jahr 0: -CapEx | Jahr 1-n: Revenue - OpEx
    cashflows = [-total_capex] + [annual_net_cashflow] * lifetime
    
    npv = npf.npv(discount_rate, cashflows)
    irr = npf.irr(cashflows)
    
    results.append({
        "Site": i + 1,
        "Dist_km": dist,
        "CapEx_M£": round(total_capex / 1e6, 2),
        "NPV_M£": round(npv / 1e6, 2),
        "IRR_%": round(irr * 100, 2),
        "LCOE_£/MWh": round(lcoe, 2)
    })

df = pd.DataFrame(results)

# =============================================================================
# 3. AUSGABE
# =============================================================================

print("--- FINANZIELLE ANALYSE (VOLLSTÄNDIG DISKONTIERT) ---")
print(df.to_string(index=False))

print("\n" + "="*50)
print(f"PORTFOLIO DURCHSCHNITT IRR: {df['IRR_%'].mean():.2f}%")
print(f"VERGLEICHSWERT WACC:       {discount_rate*100:.2f}%")
print("="*50)

--- FINANZIELLE ANALYSE (VOLLSTÄNDIG DISKONTIERT) ---
 Site  Dist_km  CapEx_M£  NPV_M£  IRR_%  LCOE_£/MWh
    1    0.466    794.95  735.65  12.78       44.10
    2    3.116    800.33  730.27  12.69       44.32
    3    2.799    799.68  730.92  12.70       44.30
    4    0.875    795.78  734.82  12.76       44.14
    5    3.586    801.28  729.32  12.67       44.36
    6    8.089    810.42  720.18  12.52       44.73
    7    1.875    797.81  732.79  12.73       44.22
    8    4.581    803.30  727.30  12.64       44.44
    9    4.736    803.61  726.98  12.63       44.45
   10    5.990    806.16  724.44  12.59       44.56
   11    7.085    808.38  722.22  12.55       44.65
   12   24.955    844.66  685.94  11.97       46.11
   13   14.071    822.56  708.03  12.32       45.22
   14    6.363    806.92  723.68  12.58       44.59
   15   26.411    847.61  682.98  11.93       46.23
   16   19.926    834.45  696.15  12.13       45.70
   17    9.122    812.52  718.08  12.48       44.81
   18    7

## Sensitivity analysis
NPV and IRR across a grid of electricity-price (£50–90/MWh) and CapEx (90–120%) scenarios.

In [10]:
import numpy as np
import pandas as pd
import numpy_financial as npf

# =============================================================================
# 1. BASIS-PARAMETER (Table 6)
# =============================================================================
capacity_mw = 500
lifetime = 35
cap_factor = 0.381
discount_rate = 0.058
base_capex_mw = 1588000
opex_mw_a = 40100
grid_cost_km = 1000000

# Wir nutzen den Durchschnitt der Top-Standorte (ca. 2.5 km)
avg_dist_km = 2.5  

def calculate_metrics(price, capex_mult):
    # CapEx Berechnung inkl. Multiplikator für Sensitivität
    current_capex = (capacity_mw * base_capex_mw * capex_mult) + (avg_dist_km * grid_cost_km)
    
    # Jährliche Cashflows
    annual_gen = capacity_mw * 8760 * cap_factor
    revenue = annual_gen * price
    opex_total = opex_mw_a * capacity_mw
    net_cashflow = revenue - opex_total
    
    # Zeitreihe für Finanzmathematik
    cashflows = [-current_capex] + [net_cashflow] * lifetime
    
    irr = npf.irr(cashflows)
    npv = npf.npv(discount_rate, cashflows)
    
    return npv / 1e6, irr * 100

# =============================================================================
# 2. SENSITIVITÄTS-SZENARIEN
# =============================================================================

# Preis-Szenarien (z.B. 60=Marktpreis niedrig, 100=starke Subvention/CfD)
prices = [50, 60, 70, 80, 90] 
# CapEx-Szenarien (0.9=günstiger, 1.2=20% Kostenüberschreitung)
capex_scenarios = [0.9, 1.0, 1.1, 1.2]

results_npv = []
results_irr = []

for p in prices:
    row_npv = {'Price_£_MWh': p}
    row_irr = {'Price_£_MWh': p}
    for c in capex_scenarios:
        npv_val, irr_val = calculate_metrics(p, c)
        col_name = f'CapEx_{int(c*100)}%'
        row_npv[col_name] = round(npv_val, 2)
        row_irr[col_name] = f"{irr_val:.2f}%"
    results_npv.append(row_npv)
    results_irr.append(row_irr)

df_npv = pd.DataFrame(results_npv)
df_irr = pd.DataFrame(results_irr)

# =============================================================================
# 3. AUSGABE
# =============================================================================

print("--- SENSITIVITÄTSANALYSE: NPV (in M£) ---")
print(df_npv.to_string(index=False))

print("\n--- SENSITIVITÄTSANALYSE: IRR (Interne Rendite) ---")
print(df_irr.to_string(index=False))

print("\n" + "="*50)
print("--- ZUSAMMENFASSUNG FÜR DEN BERICHT ---")
print(f"Hurdle Rate (Ziel-Rendite): {discount_rate*100:.2f}%")
print("Ein Projekt ist profitabel, wenn IRR > Hurdle Rate.")
print("="*50)

--- SENSITIVITÄTSANALYSE: NPV (in M£) ---
 Price_£_MWh  CapEx_90%  CapEx_100%  CapEx_110%  CapEx_120%
          50     223.90      144.50       65.10      -14.30
          60     471.63      392.23      312.83      233.43
          70     719.36      639.96      560.56      481.16
          80     967.09      887.69      808.29      728.89
          90    1214.82     1135.42     1056.02      976.62

--- SENSITIVITÄTSANALYSE: IRR (Interne Rendite) ---
 Price_£_MWh CapEx_90% CapEx_100% CapEx_110% CapEx_120%
          50     8.30%      7.28%      6.42%      5.67%
          60    10.86%      9.65%      8.64%      7.77%
          70    13.32%     11.91%     10.74%      9.74%
          80    15.73%     14.10%     12.76%     11.62%
          90    18.09%     16.26%     14.74%     13.46%

--- ZUSAMMENFASSUNG FÜR DEN BERICHT ---
Hurdle Rate (Ziel-Rendite): 5.80%
Ein Projekt ist profitabel, wenn IRR > Hurdle Rate.
